In [1]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
import os
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rank_bm25 import BM25Okapi

C:\Users\rabah\AppData\Local\Temp\ipykernel_1048\2319492943.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader


In [2]:
folder_data_path = "../data/docs"
sub_folders = [f for f in os.listdir(folder_data_path) if os.path.isdir(os.path.join(folder_data_path, f))]
print(sub_folders)
print(f"Total sub-folders found: {len(sub_folders)}")

for sub_folder in sub_folders:
    sub_folder_path = os.path.join(folder_data_path, sub_folder)
    sub_folder_pdf_files = [f for f in os.listdir(sub_folder_path) if f.endswith(".pdf")]
    sub_folder_txt_files = [f for f in os.listdir(sub_folder_path) if f.endswith(".txt")]
    print(f"Sub-folder: {sub_folder}")
    print(f"  PDF files: {sub_folder_pdf_files}")
    print(f"  TXT files: {sub_folder_txt_files}")
    print()


['General', 'IT', 'RH']
Total sub-folders found: 3
Sub-folder: General
  PDF files: ['livret_accueil.pdf', 'plan_acces.pdf', 'reglement_interieur.pdf']
  TXT files: ['horaires.txt', 'restauration.txt']

Sub-folder: IT
  PDF files: ['charte_informatique.pdf']
  TXT files: ['demande_materiel.txt', 'mot_de_passe.txt', 'securite_emails.txt', 'vpn.txt']

Sub-folder: RH
  PDF files: ['conges_absences.pdf', 'mutuelle.pdf', 'notes_de_frais.pdf', 'teletravail.pdf']
  TXT files: ['formation.txt']



## Fonctions à utiliser

In [3]:
# Nettoyage Retirer ce qui gêne : espaces, lignes vides, en-têtes répétés 
def clean_text(text: str) -> str:
    # mettre en minuscules
    text = text.lower()
    # Supprimer les espaces en début et fin de ligne
    text = text.strip()
    # Supprimer les lignes vides
    text = "\n".join([line for line in text.split("\n") if line.strip() != ""])
    # Supprimer les en-têtes répétés ("Altéa Services — Document interne — Diffusion restreinte"
    # et "Document fictif réalisé à des fins pédagogiques")
    text = re.sub(r"altéa services — document interne — diffusion restreinte", "", text)
    text = re.sub(r"document fictif réalisé à des fins pédagogiques", "", text)
    text = re.sub(r"page \d+", "", text)
    return text

# fonction pour charger les fichier txt avec DirectoryLoader
def load_txt_files(folder: str) -> list:
    path = os.path.join(folder_data_path, folder)
    loader = DirectoryLoader(path, glob="**/*.txt")
    documents = loader.load()
    for doc in documents:
        doc.page_content = clean_text(doc.page_content)
        doc.metadata["service"] = folder
        doc.metadata["page"] = doc.metadata.get("page", 0) + 1
    return documents

# Fonction pour charger les fichiers PDF avec DirectoryLoader et nettoyer leur contenu
def load_pdf_files(folder: str) -> list:
    path = os.path.join(folder_data_path, folder)
    loader = DirectoryLoader(path, glob="**/*.pdf", loader_cls=PyPDFLoader)
    documents = loader.load()

    for doc in documents:
        doc.page_content = clean_text(doc.page_content)
        doc.metadata["service"] = folder
        doc.metadata["page"] = doc.metadata.get("page", 0) + 1

    return documents

# Chunking du texte en utilisant RecursiveCharacterTextSplitter
def chunk_documents(documents: list, chunk_size: int = 1000, chunk_overlap: int = 100) -> list[dict]:
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    chunks = []
    for doc in documents:
        doc_chunks = text_splitter.split_text(doc.page_content)
        for chunk in doc_chunks:
            chunks.append({"content": chunk, "metadata": doc.metadata})
    return chunks


### Lire les fichiers pdf/txt

In [4]:
# lire les fichiers dans chaque sous-dossier du dossier de données
txt_documents = []
pdf_documents = []
for sub_folder in sub_folders:
    pdf_documents.extend(load_pdf_files(sub_folder))
    txt_documents.extend(load_txt_files(sub_folder))


Thank you for using Unstructured Open Source! Try Unstructured's commercial offering for improved output quality. 10,000 free pages to start. Learn more: https://transform.unstructured.io/?utm_source=unstructured_oss Hide this notice: UNSTRUCTURED_DISABLE_NOTICE=1
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filety

In [5]:
pdf_count = len({doc.metadata["source"] for doc in pdf_documents})
txt_count = len({doc.metadata["source"] for doc in txt_documents})

print(f"{pdf_count + txt_count} fichiers au total ({pdf_count} PDF, {txt_count} TXT)")
print(f"{len(pdf_documents)} pages PDF et {len(txt_documents)} documents TXT")

15 fichiers au total (8 PDF, 7 TXT)
16 pages PDF et 7 documents TXT


### tester plusieurs tailles de chunk

In [6]:
# tester plusieurs tailles de chunks 
# taille 200, 500, 1000
# chevauchement 20, 50, 100

chunk_sizes = [200, 500, 1000]
overlaps = 10 # 10% de chevauchement par défaut

import pandas as pd

documents = pdf_documents + txt_documents
resultats = []

for chunk_size in chunk_sizes:
    overlap = int(chunk_size * overlaps / 100)
    chunks_test = chunk_documents(
        documents,
        chunk_size=chunk_size,
        chunk_overlap=overlap,
    )

    taille_moyenne = (
        sum(len(chunk["content"]) for chunk in chunks_test) / len(chunks_test)
        if chunks_test else 0
    )

    resultats.append({
        "Taille du chunk": chunk_size,
        "Chevauchement": overlap,
        "Nombre de chunks": len(chunks_test),
        "Taille moyenne (caractères)": round(taille_moyenne, 2),
    })

df_resultats = pd.DataFrame(resultats)
display(df_resultats)
        

,Taille du chunk,Chevauchement,Nombre de chunks,Taille moyenne (caractères)
0,200,20,183,140.17
1,500,50,82,310.15
2,1000,100,46,557.89


Quel est le risque d'un chunk trop petit?
: Un chunk trop petit risque de fragmenter excessivement le texte, ce qui peut entraîner une perte de contexte et rendre la recherche moins efficace.

D'un chunk trop grand?
: Un chunk trop grand peut contenir trop d'informations, ce qui peut diluer la pertinence des résultats de recherche et augmenter le temps de traitement.

À quoi sert le chevauchement ?
: Le chevauchement permet de conserver le contexte entre les chunks consécutifs, ce qui améliore la cohérence et la pertinence des résultats de recherche.

Quelle taille retenez-vous pour la suite ?
: La taille optimale dépend du contenu et de l'application, mais une taille de chunk intermédiaire (par exemple 750 caractères) avec un chevauchement modéré (10%) est souvent un bon compromis entre contexte et granularité.

### Chunking des documents avec 750 taille et 75 chevauchements

In [7]:
# chunker les documents PDF et TXT
chunks = chunk_documents(pdf_documents + txt_documents, chunk_size=750, chunk_overlap=75)
print(f"Total chunks created: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk['content'][:100]}...")
    print(f"Metadata: {chunk['metadata']}")
    print("-" * 50)

Total chunks created: 62
Chunk 1: 1...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-11-05T22:16:55+01:00', 'author': 'Altéa Services (fictif)', 'keywords': '', 'moddate': '2026-11-05T22:16:55+01:00', 'subject': '(unspecified)', 'title': "Livret d'accueil du nouveau salarié", 'trapped': '/False', 'source': '..\\data\\docs\\General\\livret_accueil.pdf', 'total_pages': 2, 'page': 1, 'page_label': '1', 'service': 'General'}
--------------------------------------------------
Chunk 2: livret d'accueil du nouveau salarié
gen-002 — édition 2026
bienvenue chez altéa services
altéa servi...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-11-05T22:16:55+01:00', 'author': 'Altéa Services (fictif)', 'keywords': '', 'moddate': '2026-11-05T22:16:55+01:00', 'subject': '(unspecified)', 'title': "Livret d'accueil du nouveau salarié", 'trapped': '/False', 'source': '..\\dat

### Evaluer avec rank_bm25

In [8]:
tokenized_chunks = [chunk["content"].split() for chunk in chunks]
bm25 = BM25Okapi(tokenized_chunks)    

def rechercher(query: str, k: int = 2) -> list[dict]:
    query = clean_text(query)
    scores = bm25.get_scores(query.split())

    if not any(score > 0 for score in scores):
        return []

    top_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True,
    )
    return [chunks[i] for i in top_indices if scores[i] > 0][:k]

def prompt_response(query: str, k: int = 1) -> str:
    results = rechercher(query, k)

    if not results:
        return "Je ne sais pas"

    return "\n".join(
        f"[{i + 1}] {passage['content']}" for i, passage in enumerate(results)
    )

query = [
    "Combien de jours de télétravail par semaine sont autorisés, et quel jour est exclu ?",
    "Comment réinitialiser son mot de passe ?",
    "Où faut-il aller pour poser un congé ?",
    "Quels sont les plafonds de repas pour le déjeuner, le dîner et un repas d’affaires ?",
    "Quel est le délai pour remettre le bulletin d’adhésion à la mutuelle au service RH ?",
    "Que faire en cas de problème de VPN ?",
    "Comment demander un nouvel ordinateur, et quel est le délai de traitement ?",
    "Comment signaler un e-mail suspect ?",
    "À quelle heure ouvre le site de Lyon ?",
    "Où est-il permis de fumer près des locaux ?"
]
for q in query:
    tokenized_query = clean_text(q).split()
    scores = bm25.get_scores(tokenized_query)
    best_chunk_index = scores.argmax() if scores.size > 0 else -1
    print(f"Query: {q}")
    print(f"Best matching chunk index: {best_chunk_index}, score : {scores[best_chunk_index] if best_chunk_index >= 0 else 'N/A'}")
    if best_chunk_index < 0:
        print("No matching chunk found.")
    else:
        print(f"Response: {prompt_response(q)}")
    print("-" * 80)


Query: Combien de jours de télétravail par semaine sont autorisés, et quel jour est exclu ?
Best matching chunk index: 35, score : 15.033155571435671
Response: [1] télétravail
rh-004 — accord télétravail — en vigueur au 1er septembre 2025
1. qui peut télétravailler ?
le télétravail est ouvert à tous les salariés en cdi ou en cdd de plus de 6 mois, après une
période d'ancienneté de 3 mois, dès lors que leur poste le permet. les postes d'accueil, de
maintenance sur site et de logistique ne sont pas éligibles.
les alternants et les stagiaires peuvent télétravailler 1 jour par semaine, avec l'accord de
leur tuteur.
2. nombre de jours autorisés
chaque salarié éligible peut télétravailler jusqu'à 2 jours par semaine. ces jours sont fixés
avec le manager ; ils ne peuvent pas être reportés d'une semaine sur l'autre.
--------------------------------------------------------------------------------
Query: Comment réinitialiser son mot de passe ?
Best matching chunk index: 49, score : 13.400975679

### Embedding

In [9]:
# Embedding
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
embeddings = [model.encode(chunk['content']) for chunk in chunks]
print(f"Total embeddings created: {len(embeddings)}")
for i, embedding in enumerate(embeddings):
    print(f"Embedding {i+1}: {embedding[:10]}...")
    print("-" * 50)

c:\Users\rabah\Documents\insta\IA-RAG\eval\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2946.29it/s]


Total embeddings created: 62
Embedding 1: [-0.03049077  0.22248048 -0.14221765  0.04550443 -0.05430146 -0.14304778
  0.43555275  0.14467056  0.25123617 -0.22905463]...
--------------------------------------------------
Embedding 2: [-0.10251027  0.00699582 -0.18163726 -0.09946766 -0.11572463 -0.09220438
 -0.01466502  0.05931652  0.06901091  0.09973876]...
--------------------------------------------------
Embedding 3: [ 0.27935803 -0.09498412 -0.15706804 -0.2122361  -0.02068597  0.20532224
 -0.18327592 -0.00287389 -0.15809675  0.04949038]...
--------------------------------------------------
Embedding 4: [ 0.14408262  0.14345057 -0.01909844 -0.28392172  0.01358537 -0.00471571
 -0.13244385  0.2067054  -0.08029319  0.05623451]...
--------------------------------------------------
Embedding 5: [-0.03049077  0.22248048 -0.14221765  0.04550443 -0.05430146 -0.14304778
  0.43555275  0.14467056  0.25123617 -0.22905463]...
--------------------------------------------------
Embedding 6: [ 0.2189

In [10]:
# Quelle est la dimension d'un vecteur :
print(f"Dimension d'un vecteur d'embedding: {len(embeddings[0])}")
#  Combien de nombres faut-il stocker pour tout le corpus avec la taille de chunk retenue ?
print(f"Total numbers to store for the entire corpus: {len(embeddings) * len(embeddings[0])}")
print(f"Total chunks created: {len(chunks)}")
print(f"Total numbers to store per chunk: {len(embeddings[0])}")

Dimension d'un vecteur d'embedding: 384
Total numbers to store for the entire corpus: 23808
Total chunks created: 62
Total numbers to store per chunk: 384


### Indexer les chunks dans ChromaDB

In [11]:
# index dans chromaDB
import chromadb

client = chromadb.PersistentClient(path="baseVectorielle")
collection = client.get_or_create_collection("altea", metadata={"description": "Collection for Altea documents"})

texts = [chunk['content'] for chunk in chunks]
vectors = model.encode(texts, show_progress_bar=True).tolist()

collection.upsert(
    ids=[str(i) for i in range(len(texts))],
    documents=texts,
    embeddings=vectors,
    metadatas=[chunk['metadata'] for chunk in chunks]
)

print("Upsert completed successfully.")
print(f"Collection '{collection.name}' now contains {len(collection.get()['ids'])} items.")

def upsert_document_chunks(chunks_du_document: list[dict]) -> None:
    source = chunks_du_document[0]["metadata"]["source"]
    anciens_ids = set(collection.get(where={"source": source})["ids"])

    nouveaux_textes = [chunk["content"] for chunk in chunks_du_document]
    nouveaux_ids = [f"{source}::chunk::{i}" for i in range(len(chunks_du_document))]
    nouveaux_vecteurs = model.encode(nouveaux_textes).tolist()

    collection.upsert(
        ids=nouveaux_ids,
        documents=nouveaux_textes,
        embeddings=nouveaux_vecteurs,
        metadatas=[chunk["metadata"] for chunk in chunks_du_document],
    )

    ids_obsoletes = anciens_ids - set(nouveaux_ids)
    if ids_obsoletes:
        collection.delete(ids=list(ids_obsoletes))


Batches: 100%|██████████| 2/2 [00:01<00:00,  1.57it/s]

Upsert completed successfully.
Collection 'altea' now contains 62 items.


Pourquoi utilise-t-on upsert plutôt que add ? Que faudra-t-il faire le jour où un document de l'entreprise est modifié ?

On utilise upsert plutôt que add car upsert permet de mettre à jour un document existant si son ID est déjà présent dans la collection, tandis que add ne fait qu'ajouter de nouveaux documents et échoue si l'ID existe déjà.
Le jour où un document de l'entreprise est modifié, il faudra réindexer ce document en utilisant upsert avec le même ID pour mettre à jour son contenu et ses vecteurs dans la collection.


### Recherche pour voir le résultat

In [12]:
question = "Combien de télétravail par semaine ?"
resultat = collection.query(query_embeddings=model.encode([question]).tolist(), n_results=3)
print(resultat["documents"], resultat["metadatas"], resultat["distances"])

[["télétravail\nrh-004 — accord télétravail — en vigueur au 1er septembre 2025\n1. qui peut télétravailler ?\nle télétravail est ouvert à tous les salariés en cdi ou en cdd de plus de 6 mois, après une\npériode d'ancienneté de 3 mois, dès lors que leur poste le permet. les postes d'accueil, de\nmaintenance sur site et de logistique ne sont pas éligibles.\nles alternants et les stagiaires peuvent télétravailler 1 jour par semaine, avec l'accord de\nleur tuteur.\n2. nombre de jours autorisés\nchaque salarié éligible peut télétravailler jusqu'à 2 jours par semaine. ces jours sont fixés\navec le manager ; ils ne peuvent pas être reportés d'une semaine sur l'autre.", "4. indemnité et équipement\naltéa services verse une indemnité de télétravail de 2,50 € par jour télétravaillé, dans la\nlimite de 25 € par mois, avec la paie du mois suivant.\nle salarié reçoit un ordinateur portable, un casque et, sur demande, un second écran. un\nbudget de 150 € est remboursé une fois pour l'achat d'un sièg

Pourquoi le dossier base_vectorielle/ n'est-il pas envoyé sur GitHub ? Comment un collègue qui clone le
dépôt pourra-t-il la recréer ?

Le dossier base_vectorielle/ contient les données persistantes de ChromaDB, qui sont générées localement et ne doivent pas être envoyées sur GitHub pour éviter de versionner des fichiers volumineux et spécifiques à chaque machine.
Un collègue qui clone le dépôt pourra recréer ce dossier en exécutant le code d'indexation, c'est-à-dire en utilisant le client PersistentClient avec le même chemin et en upsertant les documents dans la collection.